# Notebook 1 – OSM Data Collection and Preprocessing
-----------------------------------------------------------
### Purpose

This notebook downloads the Metro Manila road network and points of interest (POIs) from OpenStreetMap. The collected datasets are cleaned, standardized, and preprocessed for subsequent feature engineering and graph-based modeling.

### Inputs

- OpenStreetMap road network
- OpenStreetMap amenities
- OpenStreetMap buildings
- OpenStreetMap shops

### Outputs

- metro_manila_processed.graphml
- road_nodes.geojson
- road_edges.geojson
- amenities.geojson
- buildings.geojson
- shops.geojson
---------------------------------------------------------------

# Section 1 - Set Up

Imports

In [ ]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from shapely.geometry import box

Configurations

In [ ]:
import sys
from pathlib import Path

# Get the project root directory 
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Study Area
METRO_MANILA = "Metro Manila, Philippines"

# Output Directories
OUTPUT_DIR = PROJECT_ROOT / "data" / "osm"

GRAPHS_DIR = OUTPUT_DIR / "graphs"
GEOJSON_DIR = OUTPUT_DIR / "geojson"
FIGURE_DIR = OUTPUT_DIR / "figures"

# Create output folders
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(exist_ok=True)
GEOJSON_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

# OSMnx Settings
ox.settings.use_cache = True
ox.settings.timeout = 600
ox.settings.log_console = False

Specify the Metro Manila's road network and the selected amenities, buildings, and shops.

In [ ]:
# Metro Manila (National Capital Region)
METRO_MANILA = "Metro Manila, Philippines"

# Official NCR Local Government Units (LGUs)
NCR_CITIES = [
    "Manila",
    "Quezon City",
    "Caloocan",
    "Las Piñas",
    "Makati",
    "Malabon",
    "Mandaluyong",
    "Marikina",
    "Muntinlupa",
    "Navotas",
    "Parañaque",
    "Pasay",
    "Pasig",
    "San Juan",
    "Taguig",
    "Valenzuela",
    "Pateros",
]

# OSM FEATURE TAGS - Selected OSM features can be configured in the config file.

# Selected amenity categories
from config.amenities import AMENITY_TAGS
from config.buildings import BUILDING_TAGS
from config.shops import SHOP_TAGS

print(f"Loaded {len(AMENITY_TAGS['amenity'])} amenity tags")
print(f"Loaded {len(BUILDING_TAGS['building'])} building tags")
print(f"Loaded {len(SHOP_TAGS['shop'])} shop tags")

# Helper Functions 

Helper functions are used to help process the data collected in this notebook. Each helper function will have a comment above it explaining its function. This section is used to help make the notebook look clean and legible.

### Download Functions

Downloads the road network of Metro Manila

In [ ]:
def download_metro_manila_road_network(network_type="drive"):
    """
    Download the Metro Manila road network.

    Parameters
    ----------
    network_type : str
        OSMnx network type (default = "drive").

    Returns
    -------
    graph : networkx.MultiDiGraph
        Raw OSM road network.

    nodes_gdf : GeoDataFrame
        Road network nodes.

    edges_gdf : GeoDataFrame
        Road network edges.
    """

    print("Downloading Metro Manila road network...")

    graph = ox.graph_from_place(
        METRO_MANILA,
        network_type=network_type,
        simplify=False,
    )

    nodes_gdf, edges_gdf = ox.graph_to_gdfs(graph)

    print(f"Nodes : {len(nodes_gdf):,}")
    print(f"Edges : {len(edges_gdf):,}")

    return graph, nodes_gdf, edges_gdf

Download the specified POI tags in Metro Manila cities.

In [ ]:
def download_osm_features(feature_name, tag_dict):
    """
    Downloads a specified OSM feature (amenities, buildings, or shops)
    for all Metro Manila cities and merges them into a single GeoDataFrame.

    Parameters
    ----------
    feature_name : str
        Name used for logging (e.g., "Amenities").

    tag_dict : dict
        Dictionary of OSM tags to download.

    Returns
    -------
    geopandas.GeoDataFrame
        Combined Metro Manila dataset.
    """

    print(f"\nDownloading {feature_name}...")

    feature_list = []

    for city in NCR_CITIES:

        city_query = f"{city}, Philippines"

        print(f"  Downloading: {city}")

        try:

            city_gdf = ox.features_from_place(
                city_query,
                tags=tag_dict,
            )

            if not city_gdf.empty:
                feature_list.append(city_gdf)

        except Exception as e:

            print(f"    Skipped ({e})")

    if not feature_list:
        print(f"No {feature_name.lower()} downloaded.")
        return gpd.GeoDataFrame()

    metro_gdf = pd.concat(
        feature_list,
        ignore_index=False,
    )

    print(f"Downloaded {len(metro_gdf):,} {feature_name.lower()}.")

    return metro_gdf

### Cleaning Functions

Removes duplicate OSM features

In [ ]:
def remove_duplicate_osm_features(gdf, dataset_name):

    before = len(gdf)

    gdf = gdf[~gdf.index.duplicated(keep="first")]

    removed = before - len(gdf)

    print(f"{dataset_name}: removed {removed:,} duplicate features.")

    return gdf

Converts OSM MultiIndex format into columns

In [ ]:
def prepare_for_export(gdf):

    if isinstance(gdf.index, pd.MultiIndex):
        gdf = gdf.reset_index()

    return gdf

Converts list and dictionary columns into strings that can be exported to GeoJSON

In [ ]:
def clean_for_geojson(gdf):

    gdf = gdf.copy()

    for column in gdf.columns:

        if column == "geometry":
            continue

        if gdf[column].apply(
            lambda x: isinstance(x, (list, dict))
        ).any():

            gdf[column] = gdf[column].astype(str)

    return gdf

Keep the specified columns for each OSM feature.

In [ ]:
def select_osm_columns(gdf):

    possible_columns = [
        "element",
        "id",
        "name",
        "amenity",
        "shop",
        "building",
        "geometry"
    ]

    existing_columns = [
        col for col in possible_columns
        if col in gdf.columns
    ]

    return gdf[existing_columns]

Simplify the road network to reduce the number of redundant nodes

In [ ]:
def simplify_road_network(graph):

    simplified = ox.simplification.simplify_graph(graph)

    return simplified

Merge nearby intersection clusters into a single node to reduce redundant nodes

In [ ]:
def consolidate_intersections(graph, tolerance=15):

    # Project to a metric CRS (meters)
    projected = ox.project_graph(
        graph,
        to_crs="EPSG:32651",   # WGS84 / UTM Zone 51N
    )


    return ox.simplification.consolidate_intersections(
        projected,
        tolerance=tolerance,
        rebuild_graph=True,
        dead_ends=False,
    )

### Export Functions

Export GeoDataFrame to GeoJSON

In [ ]:
def export_geojson(gdf, filepath):

    gdf.to_file(filepath, driver="GeoJSON")

    print(f"Saved: {filepath.name}")

# Section 2 - Data Collection

Download Metro manila Road Network

In [ ]:
metro_graph, nodes_gdf, edges_gdf = download_metro_manila_road_network()

print("\nMetro Manila road network downloaded successfully.")
print(f"Nodes : {len(nodes_gdf):,}")
print(f"Edges : {len(edges_gdf):,}")

Download specified Metro Manila OSM features

In [ ]:
amenities_gdf = download_osm_features(
    feature_name="Amenities",
    tag_dict=AMENITY_TAGS,
)

buildings_gdf = download_osm_features(
    feature_name="Buildings",
    tag_dict=BUILDING_TAGS,
)

shops_gdf = download_osm_features(
    feature_name="Shops",
    tag_dict=SHOP_TAGS,
)

print(f"Amenities : {len(amenities_gdf):,}")
print(f"Buildings : {len(buildings_gdf):,}")
print(f"Shops     : {len(shops_gdf):,}")

# Section 3 - Data Cleaning

Clean all amenities, buildings, and shops.

Since the data is download by cities, there may be some overlap between city boundaries. This then leads to duplicates. So we have to remove these duplicates.

In [ ]:
# Remove duplicates
amenities_gdf = remove_duplicate_osm_features(amenities_gdf,"Amenities")
buildings_gdf = remove_duplicate_osm_features(buildings_gdf,"Buildings")
shops_gdf = remove_duplicate_osm_features(shops_gdf,"Shops")


# Convert MultiIndex to columns
amenities_gdf = prepare_for_export(amenities_gdf)
buildings_gdf = prepare_for_export(buildings_gdf)
shops_gdf = prepare_for_export(shops_gdf)


# Remove unnecessary OSM attributes
amenities_gdf = select_osm_columns(amenities_gdf)
buildings_gdf = select_osm_columns(buildings_gdf)
shops_gdf = select_osm_columns(shops_gdf)


# Clean remaining attributes
amenities_gdf = clean_for_geojson(amenities_gdf)
buildings_gdf = clean_for_geojson(buildings_gdf)
shops_gdf = clean_for_geojson(shops_gdf)

print("\nAfter cleaning")
print(f"Amenities : {len(amenities_gdf):,}")
print(f"Buildings : {len(buildings_gdf):,}")
print(f"Shops     : {len(shops_gdf):,}")

In [ ]:
print(amenities_gdf.columns)
print(buildings_gdf.columns)
print(shops_gdf.columns)

# Section - 4 Road Network Preprocessing

Save the amount of nodes and edges before preprocessing

In [ ]:
# Before preprocessing
raw_nodes = len(metro_graph.nodes)
raw_edges = len(metro_graph.edges)

Simply the road network

In [ ]:
metro_graph = simplify_road_network(metro_graph)

simplified_nodes = len(metro_graph.nodes)
simplified_edges = len(metro_graph.edges)

Consolidate the nearby intersections (can take a while)

In [ ]:
metro_graph = consolidate_intersections(
    metro_graph,
    tolerance=15,
)

final_nodes = len(metro_graph.nodes)
final_edges = len(metro_graph.edges)

Convert back into GeoFrames

In [ ]:
nodes_gdf, edges_gdf = ox.graph_to_gdfs(metro_graph)

Convert the POI's to follow the same coordinate system (ie. EPSG:32651)

In [ ]:
amenities_gdf = amenities_gdf.to_crs(nodes_gdf.crs)
buildings_gdf = buildings_gdf.to_crs(nodes_gdf.crs)
shops_gdf = shops_gdf.to_crs(nodes_gdf.crs)

print(f"Road CRS      : {nodes_gdf.crs}")
print(f"Amenities CRS : {amenities_gdf.crs}")
print(f"Buildings CRS : {buildings_gdf.crs}")
print(f"Shops CRS : {shops_gdf.crs}")

Display preprocessing summary

In [ ]:
print("\n========== ROAD NETWORK PREPROCESSING ==========")

print(f"Raw network")
print(f"  Nodes : {raw_nodes:,}")
print(f"  Edges : {raw_edges:,}")

print(f"\nAfter simplification")
print(f"  Nodes : {simplified_nodes:,}")
print(f"  Edges : {simplified_edges:,}")

print(f"\nAfter consolidation")
print(f"  Nodes : {final_nodes:,}")
print(f"  Edges : {final_edges:,}")

print(f"\nOverall node reduction : {(raw_nodes-final_nodes)/raw_nodes*100:.1f}%")
print(f"Overall edge reduction : {(raw_edges-final_edges)/raw_edges*100:.1f}%")

# Section 5 -Exports

Export the Metro Manila graphML network

In [ ]:
ox.save_graphml(
    metro_graph,
    filepath=GRAPHS_DIR / "metro_manila_processed.graphml",
)

print("GraphML saved")

Export the road nodes into GeoJSON files

In [ ]:
export_geojson(
    nodes_gdf,
    GEOJSON_DIR / "road_nodes.geojson",

)

Export the road edges into GeoJSON files

In [ ]:
export_geojson(
    edges_gdf,
    GEOJSON_DIR / "road_edges.geojson",
)

Export the POI's (amenities, buildings, and shops) into GeoJSON files

In [ ]:
export_geojson(
    amenities_gdf,
    GEOJSON_DIR / "amenities.geojson",
)

export_geojson(
    buildings_gdf,
    GEOJSON_DIR / "buildings.geojson",
)

export_geojson(
    shops_gdf,
    GEOJSON_DIR / "shops.geojson",
)

Verification

In [ ]:
print("\n========== DATASET SUMMARY ==========")

print("Road Network")
print(f"Nodes    : {len(nodes_gdf):,}")
print(f"Edges    : {len(edges_gdf):,}")

print("\nPoints of Interest")
print(f"Amenities: {len(amenities_gdf):,}")
print(f"Buildings: {len(buildings_gdf):,}")
print(f"Shops    : {len(shops_gdf):,}")

print("\nCoordinate Reference System")
print(f"  {nodes_gdf.crs}")

print("\nOutput")
print(f"GraphML : {GRAPHS_DIR}")
print(f"GeoJSON : {GEOJSON_DIR}")